# 02 · Semi-parametric quantile regression (SPQR) for density regression

> **Source.** The code in this notebook comes from the CASE 2025 short course *AI and extremes* by **Likun Zhang, Jordan Richards and Reetam Majumder** ([Jbrich95/CASE2025_shortcourse](https://github.com/Jbrich95/CASE2025_shortcourse)). It is converted from the original `.Rmd` to a Google Colab notebook, with explanatory notes added. Changes to the original code are marked **[Adapted]** and explained where they occur.

**How to run on Google Colab.** Open this notebook in Colab, choose *Runtime → Change runtime type → R* (CPU is enough), then *Runtime → Run all*. The setup cell installs the R packages and downloads the data from the course repository. The first time Keras is used, R may spend a few minutes setting up Python/TensorFlow.

**What this notebook covers.** SPQR represents a whole conditional density as a mixture of spline basis functions, with mixture weights output by a neural network:

$$f(y \mid \mathbf x) = \sum_{k=1}^K \pi_k(\mathbf x)\, M_k(y), \qquad F(y \mid \mathbf x) = \sum_{k=1}^K \pi_k(\mathbf x)\, I_k(y),$$

where the $M_k$ are M-splines (normalised B-splines that integrate to 1), the $I_k$ are their integrals (I-splines), and $\boldsymbol\pi(\mathbf x)$ comes from a softmax layer. The loss is $-\log f(y\mid\mathbf x)$.

Background: Xu, Majumder & Reich (2022), *SPQR: An R package for semi-parametric density and quantile regression*. The data are temperature and precipitation for one US ecoregion, as used by Lawler & Shaby (2024).

In [ ]:
# [Adapted] Setup that works on Google Colab's R runtime and on a local machine.
options(timeout = 1200)                       # large data file on slow connections
on_colab <- dir.exists("/content")
if (on_colab) install.packages(c("keras3", "splines2"))
# Local machines: use a Python virtualenv called "r-keras" if one exists
if (!on_colab && reticulate::virtualenv_exists("r-keras")) {
  reticulate::use_virtualenv("r-keras", required = TRUE)
}
library(keras3)

# Course data, downloaded from the original repository if not already present
base_url <- "https://github.com/Jbrich95/CASE2025_shortcourse/raw/main/"
for (f in c("weather2.RDS")) {
  if (!file.exists(f)) download.file(paste0(base_url, f), f, mode = "wb")
}
dir.create("checkpoints", showWarnings = FALSE)   # [Adapted] folder used by the checkpoint callbacks

# If the first Keras call fails with a Python/TensorFlow error, run
#   keras3::install_keras()
# then restart the session (Runtime -> Restart session) and run this cell again.

**[Adapted]** The original also runs `remotes::install_github("rstudio/tensorflow")`. That is not needed here: `keras3` installs the released `tensorflow` R package automatically.

In [ ]:
tensorflow::set_random_seed(1)

## 1 · Key functions

* `basis()` evaluates the M-spline basis at $y$, or the I-spline basis when `integral = TRUE`, with boundary knots at 0 and 1.
* `nloglik_loss_SPQR()` is the negative log-likelihood. **The `y_true` passed to Keras is not $y$ itself but the vector of basis functions evaluated at $y$**, so $f(y\mid\mathbf x) = \sum_k \pi_k M_k(y)$ is a dot product.
* `predict.SPQR()` turns fitted weights into a PDF, a CDF, or a quantile function (QF, by interpolating the CDF).

In [ ]:
basis <- function(y, K, knots, integral = FALSE) {
  B <- splines2::mSpline(y, knots = knots,
                         Boundary.knots = c(0, 1),
                         intercept = TRUE, degree = 2,
                         integral = integral)
  return(t(B))
}

nloglik_loss_SPQR = function(y_true, y_pred) {
  numbasis <- op_shape(y_true)[[2]]
  probs <- y_pred[, 1:numbasis]
  sumprod <- op_sum(y_true * probs, axis = 2)
  spqr_loss <- -op_sum(op_log(sumprod))
  return(spqr_loss)
}

predict.SPQR <- function(model, X, Y = NULL, nY = 101, tau = 0.5, type = 'QF') {
  pred <- as.matrix(model(X))
  n <- nrow(pred)
  ntau = length(tau)
  n.knots <- ncol(pred)
  if (is.null(Y) | type == 'QF')
    Y <- seq(0, 1, length.out = nY)
  B <- (basis(Y, n.knots, knots, integral = (type != 'PDF')))
  if (ncol(B) != n)
    df1 <- pred %*% B
  if (ncol(B) == n)
    df1 <- colSums(B * t(pred))
  if (type != 'QF') {
    return(df1)
  }
  if (type == 'QF') {
    qf1 = matrix(NA, n, ntau)
    for (i in 1:n)
      qf1[i, ] <- stats::approx(df1[i, ], Y, xout = tau, ties = list("ordered", min))$y
    return(qf1)
  }
}

Apart from the network's own hyperparameters, the only tuning choice is the number of basis functions. Between 10 and 25 works well in practice.

In [ ]:
n.knots <- 15
knots = seq(1/(n.knots - 2), 1 - 1/(n.knots - 2), length = n.knots - 3)

## 2 · Data and preprocessing

Daily maximum temperature (`tmax`) is the response, and log precipitation is the covariate. `tmax` is rescaled to $[0, 1]$ because the splines live on $[0, 1]$. The data are split 80/20 at random.

> **Validation note.** The rescaling uses the range of *all* the data, including test rows. That is mild leakage. Using only the training range would need a rule for test values outside $[0, 1]$, which is itself a sign of the method's main limitation: **SPQR cannot extrapolate beyond the observed range of $y$**. The data are also daily, so a random split has the same temporal-dependence caveat as Notebook 01.

In [ ]:
weather <- readRDS("weather2.RDS")   # [Adapted] the original hidden chunk read from ../WACE_shortcourse/
head(weather)

mnth <- weather$month
tmax <- weather$tmax - 273          # Kelvin to Celsius
pr <- log(weather$pr + 0.0001)      # log precipitation
plot(tmax, pr, pch = 20)

n_total <- length(tmax)
train_ind <- sample(1:n_total, ceiling(0.8 * n_total))

tmax_range <- range(tmax)
y1 <- (tmax - tmax_range[1]) / diff(tmax_range)   # rescale to [0, 1]
X <- cbind(1, pr)

y1_train <- y1[train_ind]
y1_test <- y1[-train_ind]

X1_train <- matrix(X[train_ind, 1], ncol = 1)     # intercept only: marginal density
X1_test <- matrix(X[-train_ind, 1], ncol = 1)
X2_train <- X[train_ind, 1:2]                     # intercept + log-pr: conditional density
X2_test <- X[-train_ind, 1:2]

M_basis_train <- t(basis(y1_train, n.knots, knots))
I_basis_train <- t(basis(y1_train, n.knots, knots, integral = TRUE))
M_basis_test <- t(basis(y1_test, n.knots, knots))
I_basis_test <- t(basis(y1_test, n.knots, knots, integral = TRUE))

## 3 · Marginal distribution of `tmax`

The network's input is just an intercept, so it learns one set of 15 weights, that is, one density. A softmax output makes the weights non-negative and sum to 1.

**[Adapted]** Keras 3 discourages passing `input_shape` to a layer, so the input size is given to `keras_model_sequential()` instead. The model is the same.

In [ ]:
model <- keras_model_sequential(input_shape = c(1)) %>%
  layer_dense(units = 32, activation = 'relu') %>%
  layer_dense(units = n.knots, activation = 'softmax')

model %>% compile(optimizer = "adam", loss = nloglik_loss_SPQR)
model

early.stopping <- callback_early_stopping(monitor = "val_loss", patience = 5)
history <- model %>% fit(
  x = X1_train, y = M_basis_train,
  callbacks = list(early.stopping),
  epochs = 100, batch_size = 128, verbose = 0,
  validation_split = 0.2, shuffle = T
)

**Diagnostic.** If the fitted CDF is right, the CDF values $F(y_i)$ at test points are Uniform(0, 1), so their QQ plot against uniform quantiles should follow the diagonal.

In [ ]:
ycdf = predict.SPQR(model, X = X1_test, Y = y1_test, type = 'CDF')
qqplot(runif(length(ycdf)), ycdf)
abline(0, 1)

> **[Adapted] note.** `runif()` draws new random reference points each time, so this QQ plot changes slightly between runs. The exact uniform plotting positions `ppoints(length(ycdf))` give a deterministic version.

In [ ]:
ypred = predict.SPQR(model, X = X1_train, type = 'PDF')
plot(density(y1_train))
lines(seq(0, 1, length.out = 101), ypred[1, ], lty = 2, col = 'blue', 'l')

## 4 · Conditional distribution of `tmax` given precipitation

In [ ]:
model1 <- keras_model_sequential(input_shape = c(2)) %>%
  layer_dense(units = 32, activation = 'relu') %>%
  layer_dense(units = 15, activation = 'softmax')

model1 %>% compile(optimizer = "adam", loss = nloglik_loss_SPQR)
model1

early.stopping <- callback_early_stopping(monitor = "val_loss", patience = 5)
history <- model1 %>% fit(
  x = X2_train, y = M_basis_train,
  callbacks = list(early.stopping),
  epochs = 100, batch_size = 128, verbose = 0,
  validation_split = 0.2, shuffle = T
)

ycdf = predict.SPQR(model1, X = X2_test, Y = y1_test, type = 'CDF')
qqplot(runif(length(ycdf)), ycdf)
abline(0, 1)

Conditional densities of temperature on days with no rain (black), 1 mm (red) and 6 mm (blue).

In [ ]:
X_pred <- matrix(c(1, log(0.0001), 1, log(1), 1, log(6)), ncol = 2, byrow = T)
ypred = predict.SPQR(model1, X = X_pred, type = 'PDF')
plot(c(0:100)/100, ypred[1, ], 'l', col = 'black', xlab = 'y', ylab = 'f(y|x)', ylim = c(0, 4))
lines(c(0:100)/100, ypred[2, ], 'l', col = 'red')
lines(c(0:100)/100, ypred[3, ], 'l', col = 'blue')

Average the conditional densities over a grid of precipitation values to approximate the marginal density.

In [ ]:
pr_range <- range(pr)
pr_seq <- seq(pr_range[1], pr_range[2], length.out = 1000)
X_pred <- cbind(rep(1, 1000), pr_seq)
ypred <- predict.SPQR(model1, X = X_pred, type = 'PDF')
ypred <- apply(ypred, 2, mean)

plot(density(y1_train), ylim = c(0, 4))
lines(seq(0, 1, length.out = 101), ypred, lty = 2, col = 'blue', 'l')

> **Interpretation note.** This averages over a *uniform grid* of log-precipitation. A true marginal integrates over the *observed* distribution of precipitation, which is concentrated at zero (dry days). Averaging over the observed `X2_train` rows instead would estimate the marginal properly.

## 5 · Probability statements and quantiles

The closed-form CDF makes probability statements easy. Below: the median temperature on a dry day, and the probability that a dry day exceeds 35 °C.

In [ ]:
X_pred <- matrix(c(1, log(0.0001), 1, log(1), 1, log(6)), ncol = 2, byrow = T)
X0 <- matrix(X_pred[1, ], nrow = 1)

y0 <- predict.SPQR(model1, X = X0, type = 'QF', tau = 0.5)
y0 * diff(tmax_range) + tmax_range[1]          # median tmax (Celsius), no rain

yyy <- (35 - tmax_range[1]) / diff(tmax_range)
ycdf <- predict.SPQR(model1, X = X0, Y = yyy, type = 'CDF')
1 - ycdf                                        # P(tmax > 35 C | no rain)

## Summary

* SPQR estimates whole conditional densities with no parametric assumption, and gives PDFs, CDFs and quantiles in closed form or by interpolation.
* **Main limitation:** the M-splines are bounded on $[0, 1]$, so SPQR cannot extrapolate beyond the observed range of $y$, which is exactly where extremes live.
* **What comes next:** *SPQRx* (Majumder & Richards, 2026, *Extremes*, "Semi-parametric bulk and tail regression using spline-based neural networks") joins a GPD tail onto the SPQR bulk. Modelling the bulk and the tail jointly is the same problem as Research Question 2 of the E5 rainfall-extremes project: keeping ordinary rainfall and dry regions correct alongside the extremes.